# Bridge Stage-A Interpolant Analysis

Load a completed Stage-A-only run, inspect saved plots, and compare linear vs learned interpolant empirical W2 metrics.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ROOT


## 1) Parameters
Set `RUN_DIR` to a Stage-A-only constrained run folder under `outputs/.../constrained`.


In [ ]:
RUN_DIR = ROOT / 'outputs/2026-04-27/bridge_stagea_rescaled_ot_stage_a_only/17-00-24/constrained'
RECOMPUTE_FROM_CHECKPOINT = False
RECOMPUTE_SAMPLES = 256

RUN_DIR


## 2) Load Metrics and Config


In [ ]:
import json

metrics_path = RUN_DIR / 'metrics.json'
assert metrics_path.exists(), f'Missing metrics file: {metrics_path}'
payload = json.loads(metrics_path.read_text())
summary = payload['summary']
cfg = payload['config']

total_time = float(cfg.get('data', {}).get('bridge', {}).get('total_time', 1.0))
constraint_times = [float(t) for t in cfg.get('data', {}).get('constraint_times', [])]
constraint_mapping = [(t, round(t * total_time, 4)) for t in constraint_times]

print('mode:', summary.get('mode'))
print('coupling:', summary.get('coupling'))
print('stage_steps:', summary.get('stage_steps'))
print('cache meta:', payload.get('data_build_meta', {}))
print('bridge.total_time (physical horizon):', total_time)
print('normalized endpoint mapping: t_norm=1.0 -> t_phys=', round(total_time, 4))
print('normalized->physical constraints:', constraint_mapping)


## 3) Show Saved Stage-A Plots


In [ ]:
from IPython.display import Image, display

plot_names = [
    'training_curve.png',
    'constraint_residuals.png',
    'sample_paths.png',
    'interpolant_trajectories.png',
    'interpolant_marginal_grid.png',
    'interpolant_empirical_w2.png',
]

for name in plot_names:
    p = RUN_DIR / name
    if p.exists():
        print(f'\n{name}')
        display(Image(filename=str(p)))
    else:
        print(f'missing: {p}')


## 4) Print Linear vs Learned Interpolant W2 Table


In [ ]:
interp = summary.get('interpolant_eval', {})
linear = interp.get('linear_empirical_w2', {})
learned = interp.get('learned_empirical_w2', {})

times = sorted({float(k) for k in linear.keys()})
print('t    linear_w2    learned_w2    delta(learned-linear)')
for t in times:
    k = f'{t:.2f}'
    l = float(linear[k])
    g = float(learned[k])
    print(f'{k}  {l:10.6f}  {g:10.6f}  {g-l:+10.6f}')

print('\nlinear avg :', interp.get('linear_empirical_w2_avg'))
print('learned avg:', interp.get('learned_empirical_w2_avg'))
print('delta avg  :', interp.get('delta_avg_learned_minus_linear'))


## 5) Optional Recompute from Checkpoint
When enabled, this rebuilds bridge targets from cache, reloads `g_\theta`, recomputes interpolant metrics, and writes fresh notebook artifacts into `RUN_DIR/notebook_recompute/`.


In [ ]:
if RECOMPUTE_FROM_CHECKPOINT:
    import torch

    from cfm_project.bridge_data import prepare_bridge_problem_and_targets
    from cfm_project.data import sample_coupled_batch
    from cfm_project.metrics import interpolant_empirical_w2_metrics, interpolant_snapshot_sets
    from cfm_project.models import PathCorrection
    from cfm_project.plotting import (
        save_interpolant_marginal_comparison_grid,
        save_interpolant_trajectory_comparison,
        save_interpolant_w2_bar_plot,
    )

    device = torch.device(cfg['device'])
    prepared = prepare_bridge_problem_and_targets(
        data_cfg=cfg['data'],
        seed=int(cfg['seed']),
        device=device,
        dtype=torch.float32,
    )

    checkpoint = torch.load(RUN_DIR / 'checkpoint.pt', map_location=device)
    g_model = PathCorrection(
        state_dim=int(cfg['data']['dim']),
        hidden_dims=cfg['model']['path_hidden_dims'],
        activation=cfg['model']['activation'],
    ).to(device=device, dtype=torch.float32)
    g_model.load_state_dict(checkpoint['path_state_dict'])
    g_model.eval()

    generator = torch.Generator(device=device)
    generator.manual_seed(int(cfg['seed']))
    x0, x1, _ = sample_coupled_batch(
        prepared.problem,
        batch_size=int(RECOMPUTE_SAMPLES),
        coupling=str(cfg['data'].get('coupling', 'ot')),
        generator=generator,
    )

    interp_metrics = interpolant_empirical_w2_metrics(
        x0=x0,
        x1=x1,
        times=[float(t) for t in cfg['data']['constraint_times']],
        target_sampler=prepared.target_sampler,
        g_model=g_model,
        generator=generator,
    )
    print(interp_metrics)

    linear_by_time, learned_by_time, target_by_time = interpolant_snapshot_sets(
        x0=x0,
        x1=x1,
        times=[float(t) for t in cfg['data']['constraint_times']],
        target_sampler=prepared.target_sampler,
        g_model=g_model,
        generator=generator,
    )

    out_dir = RUN_DIR / 'notebook_recompute'
    out_dir.mkdir(parents=True, exist_ok=True)
    save_interpolant_trajectory_comparison(
        x0=x0,
        x1=x1,
        path=out_dir / 'interpolant_trajectories.png',
        g_model=g_model,
    )
    save_interpolant_marginal_comparison_grid(
        linear_samples_by_time=linear_by_time,
        learned_samples_by_time=learned_by_time,
        target_samples_by_time=target_by_time,
        path=out_dir / 'interpolant_marginal_grid.png',
    )
    save_interpolant_w2_bar_plot(
        linear_empirical_w2=interp_metrics['linear_empirical_w2'],
        learned_empirical_w2=interp_metrics['learned_empirical_w2'],
        path=out_dir / 'interpolant_empirical_w2.png',
    )
    print('wrote notebook recompute artifacts to', out_dir)
else:
    print('Set RECOMPUTE_FROM_CHECKPOINT = True to recompute metrics and plots.')
